In [1]:
env_content = "LASTFM_API_KEY=6ab2ebcbe6d3ed50f425f0c85b2793e8\nLASTFM_SECRET=7383bf8331dce497f86efd67013633c1\n"

with open('/Users/saturnine/echoes/.env', 'w') as f:
    f.write(env_content)

# verify it was created
with open('/Users/saturnine/echoes/.env', 'r') as f:
    print(f.read())

LASTFM_API_KEY=6ab2ebcbe6d3ed50f425f0c85b2793e8
LASTFM_SECRET=7383bf8331dce497f86efd67013633c1



In [2]:
import os
from dotenv import load_dotenv
load_dotenv('/Users/saturnine/echoes/.env', override=True)

print("Key:", os.getenv("LASTFM_API_KEY"))
print("Secret:", os.getenv("LASTFM_SECRET"))

Key: 6ab2ebcbe6d3ed50f425f0c85b2793e8
Secret: 7383bf8331dce497f86efd67013633c1


In [2]:
import pylast

network = pylast.LastFMNetwork(
    api_key=os.getenv("LASTFM_API_KEY"),
    api_secret=os.getenv("LASTFM_SECRET")
)

user = network.get_user("thesaturnineeee")
top = user.get_top_artists(period=pylast.PERIOD_OVERALL, limit=5)

for item in top:
    print(item.item.name, "-", item.weight, "plays")

Lana Del Rey - 8050 plays
Arctic Monkeys - 7114 plays
Taylor Swift - 2895 plays
Frank Ocean - 2811 plays
The Weeknd - 2199 plays


In [3]:
import os
from dotenv import load_dotenv
load_dotenv('/Users/saturnine/echoes/.env', override=True)

from utils.lastfm import get_artist_tags, get_similar_artists

tags = get_artist_tags("Lana Del Rey")
similar = get_similar_artists("Lana Del Rey")

print("Tags:", tags)
print("Similar:", similar)

Tags: []
Similar: []


In [4]:
code = '''import os
import pylast
import pandas as pd
from dotenv import load_dotenv

load_dotenv('/Users/saturnine/echoes/.env', override=True)

API_KEY = os.getenv("LASTFM_API_KEY")
API_SECRET = os.getenv("LASTFM_SECRET")

def get_network():
    load_dotenv('/Users/saturnine/echoes/.env', override=True)
    return pylast.LastFMNetwork(
        api_key=os.getenv("LASTFM_API_KEY"),
        api_secret=os.getenv("LASTFM_SECRET")
    )

def get_user(username):
    return get_network().get_user(username)

def get_top_artists(username, limit=20):
    user = get_user(username)
    top = user.get_top_artists(period=pylast.PERIOD_OVERALL, limit=limit)
    rows = []
    for item in top:
        artist = item.item
        try:
            image = artist.get_cover_image()
        except:
            image = ""
        rows.append({
            "name": artist.name,
            "playcount": int(item.weight),
            "url": artist.get_url(),
            "image": image,
        })
    return pd.DataFrame(rows)

def get_top_tracks(username, limit=50):
    user = get_user(username)
    top = user.get_top_tracks(period=pylast.PERIOD_OVERALL, limit=limit)
    rows = []
    for item in top:
        track = item.item
        rows.append({
            "title": track.title,
            "artist": track.artist.name,
            "playcount": int(item.weight),
            "url": track.get_url(),
        })
    return pd.DataFrame(rows)

def get_recent_tracks(username, limit=200):
    user = get_user(username)
    recent = user.get_recent_tracks(limit=limit)
    rows = []
    for track in recent:
        rows.append({
            "title": track.track.title,
            "artist": track.track.artist.name,
            "timestamp": track.timestamp,
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")
        df["hour"] = df["datetime"].dt.hour
        df["day"] = df["datetime"].dt.day_name()
        df["month"] = df["datetime"].dt.to_period("M").astype(str)
    return df

def get_artist_tags(artist_name, limit=10):
    try:
        artist = get_network().get_artist(artist_name)
        tags = artist.get_top_tags(limit=limit)
        return [t.item.name.lower() for t in tags]
    except Exception as e:
        print(f"Tags error: {e}")
        return []

def get_similar_artists(artist_name, limit=10):
    try:
        artist = get_network().get_artist(artist_name)
        similar = artist.get_similar(limit=limit)
        return [s.item.name for s in similar]
    except Exception as e:
        print(f"Similar error: {e}")
        return []

def get_similar_users(username, limit=10):
    try:
        user = get_user(username)
        neighbors = user.get_neighbours(limit=limit)
        return [n.item.name for n in neighbors]
    except Exception as e:
        print(f"Similar users error: {e}")
        return []

def get_user_info(username):
    try:
        user = get_user(username)
        return {
            "username": username,
            "playcount": user.get_playcount(),
            "registered": user.get_registered(),
            "image": user.get_image(),
            "url": user.get_url(),
        }
    except Exception as e:
        print(f"User info error: {e}")
        return {"username": username}

def build_user_profile(username):
    print(f"Fetching profile for {username}...")
    return {
        "info":        get_user_info(username),
        "top_artists": get_top_artists(username, limit=20),
        "top_tracks":  get_top_tracks(username, limit=50),
        "recent":      get_recent_tracks(username, limit=200),
    }
'''

with open('/Users/saturnine/echoes/utils/lastfm.py', 'w') as f:
    f.write(code)

print("Written!")

Written!


In [1]:
from utils.lastfm import get_artist_tags, get_similar_artists

tags = get_artist_tags("Lana Del Rey")
similar = get_similar_artists("Lana Del Rey")

print("Tags:", tags)
print("Similar:", similar)

Tags: ['indie', 'female vocalists', 'indie pop', 'pop', 'alternative', 'dream pop', 'american', 'singer-songwriter', 'trip-hop', 'sadcore']
Similar: ['Emile Haynie', 'Melanie Martinez', 'Lorde', 'Saint Avangeline', 'Marina & the Diamonds', 'Billie Eilish', 'Marina', 'Remy Bond', 'Fiona Apple', 'Sky Ferreira']


In [2]:
from utils.lastfm import build_user_profile, get_artist_tags, get_similar_artists

# build your full profile
profile = build_user_profile("thesaturnineeee")

# get tags for your top 5 artists
print("=== ARTIST TAGS ===")
for artist in profile["top_artists"]["name"].head(5):
    tags = get_artist_tags(artist)
    print(f"{artist}: {tags}")

Fetching profile for thesaturnineeee...


/Users/saturnine/echoes/utils/lastfm.py:65: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")


=== ARTIST TAGS ===
Lana Del Rey: ['indie', 'female vocalists', 'indie pop', 'pop', 'alternative', 'dream pop', 'american', 'singer-songwriter', 'trip-hop', 'sadcore']
Arctic Monkeys: ['indie rock', 'indie', 'british', 'alternative', 'rock', 'alternative rock', 'britpop', 'garage rock', 'post-punk revival', 'post-punk']
Taylor Swift: ['country', 'pop', 'female vocalists', 'singer-songwriter', 'acoustic', 'country pop', 'taylor swift', 'american', 'snake', 'alternative']
Frank Ocean: ['rnb', 'soul', 'hip-hop', 'r&b', 'ofwgkta', 'neo-soul', 'alternative rnb', 'american', 'hip hop', 'pop']
The Weeknd: ['rnb', 'electronic', 'dubstep', 'canadian', 'prog-rnb', 'pop', 'r&b', 'alternative rnb', 'soul', 'hip-hop']


In [3]:
# find this line in utils/lastfm.py and replace it
# OLD:
# df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")

# Run this to fix it:
with open('/Users/saturnine/echoes/utils/lastfm.py', 'r') as f:
    content = f.read()

content = content.replace(
    'df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")',
    'df["datetime"] = pd.to_datetime(pd.to_numeric(df["timestamp"], errors="coerce"), unit="s")'
)

with open('/Users/saturnine/echoes/utils/lastfm.py', 'w') as f:
    f.write(content)

print("Fixed!")

Fixed!
